# R06 · Information-budget and proxy comparison

Compare learned assessment with a simple proxy and actual byte-matched finer summaries. A negative comparison is a necessary control for the proposed information cost.

This notebook reuses the **completed notebook R05** models, predictions and raw2 payloads. **No model is fitted, no gate
is recalibrated, and notebook R05 training/preparation is never executed.** The existing test remains exploratory.

Two questions are kept separate:

1. With the **same original histogram answer**, does ML add value beyond a simple disagreement formula using the same raw2 information?
2. With the **same serialized budget**, is it better to return an answer from a finer histogram instead?

## Run on Databricks

1. Pull this branch in your personal Git folder; attach the same runtime used for 15. Open **R06**, not the training notebook.
2. Run the configuration cell. Normally leave **`run_dir` blank**: it selects the latest complete `within-source-...` run
   under your personal `output_root`, by modification time only. For a specific source, use its complete R05 result directory.
   Do not enter a ZIP, a notebook path, `error-objective-...` or `full-query-report-...` here.
3. Leave `raw_data_dir`, `queries_file` and `logger_file` blank when the source inputs have not moved. Raw files must
   retain the exact compressed/uncompressed identities recorded by 15. Relocation overrides still require matching hashes.
4. Keep `seeds=0,1,2`, or use `seeds=0` for a small first check. `timing_queries=30` and `timing_repeats=5` limit only
   runtime sampling; **quality uses the entire frozen test set** for every requested seed.
5. Run All. Missing LightGBM is installed at the source R05 version. Other runtime versions must match. The original R01
   input manifest, query file and logger remain read-only provenance dependencies. Saved R05 model/payload files are required.
6. Share **`information_budget_results.zip`** from the printed new `information-budget-...` directory. Previous results
   are never overwritten. Failed runs preserve available diagnostics in the same ZIP format.

## Fixed-answer methods

| Method | Returned answer | Reliability assessment |
|---|---|---|
| `base_sa` | Original H | Faithful original SA formula, checked against saved scores |
| `raw2_ml` | Same H | Saved R05 raw2 LightGBM; live predictions are checked during timing |
| `raw2_proxy` | Same H | `abs(H-Sfine)/max(Sfine,1)` |
| `coarse_proxy` | Same H | `abs(H-Scoarse)/max(Scoarse,1)`; useful lower-information reference |

`Sfine` is the sum of raw2 subhistogram estimates; `Scoarse` uses the original source-cell totals recovered exactly from
children. These are approximate answers, not truth. The formula is fixed without looking at test outcomes. Cases with
zero auxiliary estimate are exported; a zero score is not a safety guarantee. ML and proxy numerical prediction MAE use
the same actual error of H. SA remains a heuristic score, so its numerical MAE is undefined.

## Stronger finer-answer controls

| Method | How the budget is defined |
|---|---|
| `fine_synopsis_sa` | Same bytes as original H plus all raw2 payloads; model excluded on both sides |
| `fine_total_sa` | Same bytes as H plus raw2 plus the **matching saved model file for each seed** |

Each control builds a NEW uniform **square** grid by counting the same raw points within the original base extent.
It uses the same uncompressed, lossless uint32 payload format as raw2 plus an 8-byte persisted SA alpha. The largest
fitting square resolution is selected by actual file sizes, without query rectangles, truth or scores. The next larger
square must exceed the cap. This fills the budget much more closely than selecting from sparse existing ARM exports.
The new control is **not** an ARM algorithm export or a proof of the best possible finer histogram.

Fine methods return a different answer and have their own actual error. Compare actual answer quality, selected-answer
risk, space and time; do not rank different-target prediction MAEs as if the evaluator task were unchanged. Direct
`Sfine`/`Scoarse` answers are also reported for answer quality, without inventing a score or deployment rule for them.

Model weights and SA parameters are included explicitly. Python/native process memory, derived arrays, construction,
initialization and training are different costs. The raw2 model is not free. There is no 5% acceptance-risk hard target,
no new operational cutoff, no exact fallback, and no E2E speedup claim.

In [ ]:
import ast
import base64
import gzip
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
import platform
import shutil
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd

EPSILON = .25
FRACTIONS = (.10, .25, .50, 1.)
METHODS = ('base_sa', 'raw2_ml', 'raw2_proxy', 'coarse_proxy', 'fine_synopsis_sa', 'fine_total_sa')


def notebook_defaults():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        root = ctx.notebookPath().get().split('/ml/notebooks/')[0]
        repo = root if root.startswith('/Workspace/') else '/Workspace'+root
        output = '/Workspace/Users/'+ctx.userName().get()+'/synopsislake-results'
    except (NameError, AttributeError):
        repo, output = str(Path.cwd()), str(Path(tempfile.gettempdir())/'synopsislake-results')
    return dict(repo_root=repo, output_root=output, run_dir='', raw_data_dir='', queries_file='', logger_file='',
                seeds='0,1,2', timing_queries='30', timing_repeats='5')


DEFAULTS = notebook_defaults()
try:
    for key, value in DEFAULTS.items():
        dbutils.widgets.text(key, value)
    get_config = dbutils.widgets.get
except NameError:
    def get_config(key):
        return os.environ.get(key.upper(), DEFAULTS[key])


def load_settings():
    cfg = {k:str(get_config(k)).strip() for k in DEFAULTS}
    cfg['seeds'] = [int(x) for x in cfg['seeds'].split(',')]
    if not cfg['seeds'] or len(set(cfg['seeds'])) != len(cfg['seeds']):
        raise ValueError('Use distinct seeds present in the completed notebook 15 run')
    for key in ('timing_queries','timing_repeats'):
        cfg[key] = int(cfg[key])
        if cfg[key] < 1:
            raise ValueError(key+' must be positive')
    return cfg


def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()


def save_json(path, value):
    path=Path(path);temporary=path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value,indent=2,allow_nan=False,default=str));temporary.replace(path)


def choose_ids(ids, count, salt):
    return sorted(sorted(set(map(int,ids)),key=lambda q:hashlib.sha256(f'{salt}:{q}'.encode()).digest())[:count])


def resolve_source(cfg):
    if cfg['run_dir']:
        source=Path(cfg['run_dir']).expanduser().resolve()
    else:
        candidates=[]
        for path in Path(cfg['output_root']).expanduser().glob('*/run.json'):
            info=json.loads(path.read_text())
            if (info.get('status')=='complete' and info.get('study')=='within-source-information'
                    and set(cfg['seeds'])<=set(info.get('config',{}).get('seeds',[]))):
                candidates.append(path)
        if not candidates:
            raise FileNotFoundError('Set run_dir to the completed notebook 15 within-source-... result directory, not a ZIP or notebook path.')
        source=max(candidates,key=lambda p:p.stat().st_mtime_ns).parent
    prior=json.loads((source/'run.json').read_text())
    if (prior.get('status')!='complete' or prior.get('study')!='within-source-information'
            or not set(cfg['seeds'])<=set(prior['config']['seeds'])):
        raise ValueError('run_dir must contain a completed notebook 15 experiment with the requested seeds')
    print('Using completed notebook 15:',source,flush=True)
    return source,prior

In [ ]:
# INSTALL_DEPENDENCIES
import importlib.util
_config_for_install = load_settings()
_source_for_install, _prior_for_install = resolve_source(_config_for_install)
if importlib.util.find_spec('lightgbm') is None:
    get_ipython().run_line_magic('pip', 'install lightgbm==' + _prior_for_install['package_versions']['lightgbm'])
else:
    print('LightGBM is available; its version will be checked against the completed source run.')

## Reuse verified definitions

Only pinned definitions from 15/R03/R02 are loaded, never their training, preparation, widgets or report entry points.
AST checks tolerate presentation changes and missing Databricks cell IDs, but reject changed definitions. If a notebook
is not a readable workspace file, read-only Workspace export uses the signed-in runtime identity, without a personal token.

In [ ]:
REUSE_SPECS_15 = {'within-source-04': None, 'within-source-06': ['save_raw_payload', 'load_raw_payload', 'iter_raw_points', 'coarse_indices'], 'within-source-08': ['prediction_metrics', 'risk_coverage_rows']}
REUSE_HASHES_15 = {'within-source-04': '2140517568e7336df641bdfdb715cc3223cdfe6c94a6e4c67b9139cdc6431be0', 'within-source-06': 'c772ae0afd9e53fb6faa5178560ae95f3b689e02ba60eb6b74369609db17eda4', 'within-source-08': '30b2610208309378a5ade1a006864b37fe2c08eb00891beb54eff51dbd08fd36'}


def load_verified_helpers(repo):
    path=Path(repo).resolve()/'ml/notebooks/17_within_source_information.ipynb'
    try:
        raw=path.read_bytes();document=json.loads(raw);method='workspace filesystem'
    except (OSError,ValueError,UnicodeError):
        if not str(path).startswith('/Workspace/'):
            raise
        from databricks.sdk import WorkspaceClient
        from databricks.sdk.service.workspace import ExportFormat
        client=WorkspaceClient();last=None
        for candidate in (str(path)[len('/Workspace'):],str(path)[len('/Workspace'):].removesuffix('.ipynb')):
            try:
                exported=client.workspace.export(path=candidate,format=ExportFormat.JUPYTER)
                raw=base64.b64decode(exported.content);document=json.loads(raw);method='Workspace API using runtime identity';break
            except Exception as error:
                last=error
        else:
            raise ValueError('Cannot read notebook 15; check repo_root and workspace permissions: '+str(last)) from last
    env=dict(globals());env['__name__']='verified_notebook15_definitions'
    # These names are used by the reused definition loaders, never by an experiment entry point.
    env.update(INNER_FRACTION=.10,EARLY_STOPPING_ROUNDS=100)
    for label,names in REUSE_SPECS_15.items():
        matches=[]
        for cell in document['cells']:
            if cell['cell_type']!='code':continue
            try:tree=ast.parse(''.join(cell['source']))
            except SyntaxError:continue
            if names is not None:
                nodes=[n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name in names]
                if {n.name for n in nodes}!=set(names):continue
                tree=ast.Module(body=nodes,type_ignores=[])
            elif any(not isinstance(n,(ast.Import,ast.ImportFrom,ast.Assign,ast.FunctionDef)) for n in tree.body):continue
            if hashlib.sha256(ast.dump(tree,include_attributes=False).encode()).hexdigest()==REUSE_HASHES_15[label]:
                matches.append(tree)
        if len(matches)!=1:
            raise ValueError('Notebook 15 definitions changed/missing: '+label+'. Pull matching notebooks 12, 13, 15 and 16.')
        exec(compile(matches[0],str(path)+':'+label,'exec'),env)
    parent,parent_provenance=env['load_notebook13'](repo)
    namespace,compact_provenance=parent['load_reused_definitions'](repo)
    return env,namespace,dict(notebook15=dict(path=str(path),read_method=method,document_sha256=hashlib.sha256(raw).hexdigest(),
        definition_ast_sha256=REUSE_HASHES_15),notebook13=parent_provenance,notebook12=compact_provenance)

## Faithful live SA and a cheap non-ML path

SA fits a symmetric beta distribution using median/MAD of density, including zero cells, and averages per-cell beta-tail
error by **area-overlap fraction**, including empty cells. Original base alpha comes from the hash-verified logger.
The live base estimate and SA scores must reproduce saved R05 outputs. New square grids avoid the repository's legacy
rectangular density-stride issue. If fine grids create large alpha or tied scores, retain and report that behavior.

Simple proxies call a dedicated count-only estimator. They do not pay for the seven ML source features, and no runtime
provider receives truth, saved predictions or raw point files. Saved values are used only for post-timer parity checks.

In [ ]:
"""Count-only estimation and the repository's Skewness-Align query score.

These functions use the same positive-area overlap semantics as features.py and
SimpleSpatialHistogramOpt.RangeQueryWithRatio. SA is an area-fraction-weighted
heuristic, not a prediction expressed in relative-error percentage units.

sa_alpha implements the mathematical density flattening (correct row stride).
The original logger's legacy stride implementation is equivalent on the square
ARM grids and the square finer grids used in this comparison. It is not a
promise to reproduce the legacy rectangular-grid stride bug.
"""
import numpy as np
from scipy.special import betainc


def _sa_hist_parts(hist):
    if len(hist) != 4:
        raise ValueError('Histogram must be (rate, bx, by, counts)')
    _, bx, by, counts = hist
    bx, by, counts = np.asarray(bx), np.asarray(by), np.asarray(counts)
    if (bx.ndim != 1 or by.ndim != 1 or len(bx) < 2 or len(by) < 2 or
            counts.ndim != 2 or counts.shape != (len(bx)-1, len(by)-1)):
        raise ValueError('Invalid histogram dimensions')
    return bx, by, counts


def sa_alpha(hist):
    """Compute target-histogram beta shape once at initialization, without sources.

    Constants and the median/MAD convention match CountHistExp/ErrorPredictionLogExp.
    Zero-density cells are retained. All-empty or zero-median/zero-MAD grids
    legitimately produce alpha=500001 and may make SA scores heavily tied.
    """
    bx, by, counts = _sa_hist_parts(hist)
    if (not np.isfinite(bx).all() or not np.isfinite(by).all() or
            not np.isfinite(counts).all() or (counts < 0).any() or
            (np.diff(bx) <= 0).any() or (np.diff(by) <= 0).any()):
        raise ValueError('Invalid histogram boundaries or counts')
    density = counts / np.outer(np.diff(bx), np.diff(by))
    median = float(np.median(density))
    mad = float(np.median(np.abs(density-median)))
    alpha = .5 / (mad/(median+1e-6) + 1e-6) + 1.
    if not np.isfinite(alpha) or alpha <= 0:
        raise ValueError('Invalid SA beta shape')
    return float(alpha)


def _sa_overlap(boundaries, lower, upper):
    # Exact formula from features._overlap; no target-wide feature statistics.
    n = len(boundaries)-1
    i0 = int(np.clip(np.searchsorted(boundaries, lower, 'right')-1, 0, n-1))
    i1 = int(np.clip(np.searchsorted(boundaries, upper, 'left')-1, 0, n-1))
    left, right = boundaries[i0:i1+1], boundaries[i0+1:i1+2]
    fraction = np.clip((np.minimum(upper, right)-np.maximum(lower, left))/(right-left), 0, 1)
    return i0, i1, fraction


def _sa_query_overlap(q, hist):
    """Return positive-area geometry plus aligned counts; hist is setup-validated.

    Per-query code intentionally avoids scanning/revalidating the entire grid or
    allocating a float64 copy of every count. Query construction never receives
    truth, training labels, a source file path or raw points.
    """
    bx, by, counts = _sa_hist_parts(hist)
    q = np.asarray(q, dtype=float)
    if q.shape != (4,) or not np.isfinite(q).all() or q[0] >= q[1] or q[2] >= q[3]:
        raise ValueError('Query must contain four finite increasing rectangle bounds')
    # Boundary-only contact has zero area and contributes neither counts nor SA.
    if q[1] <= bx[0] or q[0] >= bx[-1] or q[3] <= by[0] or q[2] >= by[-1]:
        return np.empty((0, 0)), np.empty((0, 0))
    i0, i1, fx = _sa_overlap(bx, q[0], q[1])
    j0, j1, fy = _sa_overlap(by, q[2], q[3])
    weight = np.outer(fx, fy)
    return weight, counts[i0:i1+1, j0:j1+1]


def histogram_count(q, hist):
    """Uniform-area range-count estimate only, without ML/source feature work."""
    weight, counts = _sa_query_overlap(q, hist)
    return float(np.sum(weight*counts))


def query_answer_sa(q, hist, alpha):
    """Return (count estimate, original SA heuristic) in one geometry traversal.

    All positive-overlap cells participate in SA, INCLUDING empty cells. Each
    cell's weight is its area-overlap fraction divided by the sum of fractions,
    never its point count. This is not a certified numerical error estimate.
    """
    alpha = float(alpha)
    if not np.isfinite(alpha) or alpha <= 0:
        raise ValueError('SA alpha must be finite and positive')
    weight, counts = _sa_query_overlap(q, hist)
    estimate = float(np.sum(weight*counts))
    ratios = weight[weight > 0]
    if not len(ratios):
        return estimate, 0.
    errors = 1.-betainc(alpha, alpha, .5+ratios/2)+betainc(alpha, alpha, .5-ratios/2)
    score = float(np.sum(ratios*errors)/np.sum(ratios))
    if not np.isfinite(estimate) or not np.isfinite(score):
        raise ValueError('Nonfinite histogram estimate or SA score')
    return estimate, score

## Freeze the cohort, models and input identities

All methods share each seed's exact outer test membership. Source-cell grouping is inherited, not a claim of strict
non-overlapping geographic or cross-dataset transfer. Raw payloads and models are verified by saved hashes; source
children must reproduce the original parent payload. Query parsing matches the training pipeline.
Saved R05 parity audits must have passed. Truth below 10, including zero, remains excluded.

In [ ]:
def verified_path(path, expected, label):
    path=Path(path).expanduser().resolve()
    if not path.is_file():raise FileNotFoundError('Missing '+label+': '+str(path))
    if sha(path)!=expected:raise ValueError(label+' changed: '+str(path))
    return path


def load_inputs(cfg,source,prior,destination,helpers,namespace,state):
    versions={n:importlib.metadata.version(n) for n in prior['package_versions']}
    if versions!=prior['package_versions']:
        raise ValueError('Use the same runtime package versions as notebook 15: '+str(versions))
    for name,digest in prior['source_code_sha256'].items():
        if sha(Path(cfg['repo_root'])/'ml'/name)!=digest:
            raise ValueError('Inherited feature/helper code changed: '+name)
    parent=verified_path(Path(prior['source_run'])/'run.json',prior['source_manifest_sha256'],'original 08d manifest')
    original=json.loads(parent.read_text())
    manifest=json.loads((source/'preparation/input_manifest.json').read_text())
    for key,item in manifest.items():
        if item['sha256']!=original['input_sha256'][key]:
            raise ValueError('Saved 15 input identity differs from its original source: '+key)
    inputs={}
    for key,widget in [('data/queries-Ans.txt','queries_file'),('run/rows.csv','logger_file')]:
        item=manifest[key];path=cfg[widget] or item['path']
        inputs[key]=verified_path(path,item['sha256'],key)
    queries=pd.read_csv(inputs['data/queries-Ans.txt'],header=None).to_numpy(float)
    if queries.ndim!=2 or queries.shape[1]!=5 or not np.isfinite(queries).all():raise ValueError('Invalid query input')
    outer=pd.read_csv(source/'outer_splits.csv');outer=outer[outer.seed.isin(cfg['seeds'])].copy()
    if sha(source/'outer_splits.csv')!=prior['source13_artifact_sha256']['outer_splits.csv']:
        raise ValueError('Notebook 15 outer split identity changed')
    if outer.duplicated(['seed','qId']).any():raise ValueError('Duplicate outer split IDs')
    for seed in cfg['seeds']:
        part=outer[outer.seed==seed]
        if set(part.split)!={'train','calibration','test'}:raise ValueError('Incomplete outer roles')
        groups=[set(part.loc[part.split==r,'cell']) for r in ('train','calibration','test')]
        if any(groups[i]&groups[j] for i in range(3) for j in range(i)):raise ValueError('Overlapping outer cell groups')
    saved=pd.read_csv(source/'test_predictions.csv')
    saved=saved[saved.seed.isin(cfg['seeds'])].copy()
    if saved.duplicated(['seed','variant','base_rate','qId']).any():raise ValueError('Duplicate saved prediction keys')
    model_rows=pd.read_csv(source/'models.csv');model_rows=model_rows[(model_rows.variant=='raw2_source')&model_rows.seed.isin(cfg['seeds'])]
    definitions=json.loads((source/'feature_definitions.json').read_text())
    names=definitions['raw2_source']
    if len(names)!=21 or any(n in names for n in ('skew_pred','log_alpha','truth','relErr','qId')):
        raise ValueError('Expected notebook 15 no-SA 21-feature model')
    stage=destination/'payloads';stage.mkdir()
    models={};model_paths={}
    for seed in cfg['seeds']:
        row=model_rows[model_rows.seed==seed]
        if len(row)!=1:raise ValueError('Missing/duplicate saved raw2 model')
        path=verified_path(source/'models'/f'seed_{seed}'/'raw2_source'/'model.txt',row.iloc[0].model_sha256,'saved raw2 model')
        if path.stat().st_size!=int(row.iloc[0].model_bytes):raise ValueError('Saved model byte count mismatch')
        tick=time.perf_counter();models[seed]=lgb.Booster(model_file=str(path));model_paths[seed]=path
        if models[seed].feature_name()!=names:raise ValueError('Saved model feature order differs from source15 schema')
        state['initialization'].append(dict(method='raw2_ml',seed=seed,kind='model_load',seconds=time.perf_counter()-tick))
    raw_manifest=json.loads((source/'raw_input_manifest.json').read_text())
    raw_hists=[];coarse_hists=[];raw_payload_paths=[];coarse_paths=[]
    for record in raw_manifest:
        original_path=Path(record['payload_path'])
        path=source/'raw_payloads'/original_path.name
        if not path.is_file():path=original_path
        verified_path(path,record['payload_sha256'],'raw2 payload')
        copied=stage/path.name;shutil.copyfile(path,copied)
        fine=helpers['load_raw_payload'](copied);raw_hists.append(fine);raw_payload_paths.append(copied)
        nx,ny=fine[3].shape
        if nx%2 or ny%2:raise ValueError('Expected 2x2 subdivisions of each source')
        parent_counts=fine[3].reshape(nx//2,2,ny//2,2).sum(axis=(1,3))
        coarse=(fine[0],fine[1][::2],fine[2][::2],parent_counts);coarse_hists.append(coarse)
        coarse_path=stage/f"src_{record['source_id']}-coarse.hist"
        namespace['save_hist'](coarse_path,coarse);coarse_paths.append(coarse_path)
        old=source/'preparation/payloads'/f"src_{record['source_id']}.hist"
        if old.is_file() and sha(old)!=sha(coarse_path):raise ValueError('Raw children do not reproduce original coarse source payload')
    rates=sorted(saved.base_rate.unique());base_hists={};base_paths={};alphas={};alpha_paths={}
    logger=pd.read_csv(inputs['run/rows.csv'],usecols=['method','mergeRate','betaAlpha'])
    for rate in rates:
        part=saved[np.isclose(saved.base_rate,rate,rtol=0,atol=1e-12)]
        keys=part.histogram_key.unique()
        if len(keys)!=1:raise ValueError('Expected one fixed base histogram per base rate')
        key=str(keys[0]);p=source/'preparation/payloads'/(key+'.hist')
        hist=namespace['load_hist_payload'](p)
        source_key='run/'+key+'.txt'
        old=verified_path(manifest[source_key]['path'],manifest[source_key]['sha256'],'original histogram export')
        exported=load_export_hist(old)
        if not all(np.array_equal(a,b) for a,b in zip(hist[1:],exported[1:])):raise ValueError('Base binary differs from original export')
        copied=stage/(key+'.hist');shutil.copyfile(p,copied)
        base_hists[rate]=hist;base_paths[rate]=copied
        alpha_rows=logger[(logger.method=='ARM')&np.isclose(logger.mergeRate,rate,rtol=0,atol=1e-12)].betaAlpha.to_numpy()
        if not len(alpha_rows) or not np.isfinite(alpha_rows).all() or not np.allclose(alpha_rows,alpha_rows[0],rtol=1e-12,atol=1e-12):
            raise ValueError('Missing/inconsistent original SA alpha')
        alpha=float(alpha_rows[0])
        if alpha<=0:raise ValueError('Invalid original SA alpha')
        alphas[rate]=alpha;ap=stage/(key+'.alpha');np.asarray([alpha],dtype='<f8').tofile(ap);alpha_paths[rate]=ap
        if hist[3].shape[0]==hist[3].shape[1] and not np.isclose(sa_alpha(hist),alpha,rtol=1e-10,atol=1e-10):
            raise ValueError('Original square-grid beta fit does not reproduce')
    if len(raw_hists)!=len(raw_manifest):raise ValueError('Missing raw2 sources')
    payload_table=pd.read_csv(source/'payloads.csv')
    for rate in rates:
        row=payload_table[(payload_table.variant=='raw2_source')&np.isclose(payload_table.base_rate,rate,rtol=0,atol=1e-12)]
        total=base_paths[rate].stat().st_size+sum(p.stat().st_size for p in raw_payload_paths)
        if len(row)!=1 or total!=int(row.iloc[0].total_synopsis_bytes):raise ValueError('raw2 budget differs from notebook 15')
    # Saved source 15 audits must have succeeded, independently of its complete status flag.
    for name,column in [('baseline_reproduction.csv','scores_and_targets_match'),('raw_truth_parity.csv','matches'),
                        ('raw_source_parity.csv','subcell_mass_conserved')]:
        audit=pd.read_csv(source/name)
        if audit.empty or not audit[column].astype(str).str.lower().eq('true').all():raise ValueError('Source15 audit failed: '+name)
    if not pd.read_csv(source/'raw_source_parity.csv').max_coarse_count_gap.eq(0).all():raise ValueError('Source15 coarse parity failed')
    outer.to_csv(destination/'outer_splits.csv',index=False)
    save_json(destination/'feature_definitions.json',definitions)
    state.update(package_versions=versions,source15_run=str(source),source15_manifest_sha256=sha(source/'run.json'),
        source15_artifact_sha256={n:sha(source/n) for n in ('test_predictions.csv','models.csv','outer_splits.csv',
            'raw_input_manifest.json','feature_definitions.json','payloads.csv')},
        verified_inputs={k:dict(path=str(p),sha256=sha(p)) for k,p in inputs.items()})
    return dict(queries=queries,outer=outer,saved=saved,rates=rates,models=models,model_paths=model_paths,names=names,
        base_hists=base_hists,base_paths=base_paths,alphas=alphas,alpha_paths=alpha_paths,
        raw_hists=raw_hists,raw_paths=raw_payload_paths,coarse_hists=coarse_hists,coarse_paths=coarse_paths,
        raw_manifest=raw_manifest,model_rows=model_rows)


def load_export_hist(path):
    lines=Path(path).read_text().splitlines()
    return (float(lines[0]),np.asarray(lines[1].split(','),float),np.asarray(lines[2].split(','),float),
            np.asarray([r.split(',') for r in lines[3:]],float))

## Build finer histograms independently of queries

The raw pass simultaneously fills all distinct budget-derived grids and verifies each original raw2 source subcell count.
No point is silently dropped, and every finer grid must conserve the same raw mass. Raw identities are checked before
and after construction. Raw construction is offline and separately timed; it never occurs inside the query timer.

Both synopsis-only and total-budget views are reported. Different seeds may select different total-budget resolutions
because their model sizes differ. Grids that resolve to the same physical representation are built only once.

In [ ]:
def largest_square_grid(cap,bounds,save_payload,folder):
    # All raw counts fit uint32 because the total raw row count is checked before planning.
    # Probe real uncompressed archive bytes; headers/boundaries/alpha are included, not estimated away.
    with tempfile.TemporaryDirectory(prefix='budget-probe-',dir=str(folder)) as temporary:
        path=Path(temporary)/'probe.npz'
        def size(n):
            bx=np.linspace(bounds[0],bounds[1],n+1);by=np.linspace(bounds[2],bounds[3],n+1)
            save_payload(path,(1.,bx,by,np.zeros((n,n),np.uint32)))
            return path.stat().st_size+8
        if size(1)>cap:raise ValueError('Budget too small for one finer cell and its SA parameter')
        low,high=1,max(2,int(math.sqrt(cap/4))+2)
        while low+1<high:
            mid=(low+high)//2
            if size(mid)<=cap:low=mid
            else:high=mid
        return low,size(low),size(low+1)


def build_finer_baselines(cfg,data,destination,helpers,namespace,state):
    audit=pd.read_csv(Path(state['source15_run'])/'raw_source_parity.csv')
    expected_total=int(audit.raw_rows.sum())
    if expected_total>np.iinfo(np.uint32).max:
        raise ValueError('This comparison requires raw row totals fitting uint32; select an explicit uint64 budget protocol for larger data')
    folder=destination/'finer_payloads';folder.mkdir()
    plans=[];grids={}
    for rate in data['rates']:
        hist=data['base_hists'][rate];bounds=(hist[1][0],hist[1][-1],hist[2][0],hist[2][-1])
        synopsis=data['base_paths'][rate].stat().st_size+sum(p.stat().st_size for p in data['raw_paths'])
        caps=[('synopsis',None,synopsis)]
        caps.extend(('total',seed,synopsis+data['model_paths'][seed].stat().st_size) for seed in cfg['seeds'])
        for scope,seed,cap in caps:
            n,planned,next_bytes=largest_square_grid(int(cap),bounds,helpers['save_raw_payload'],folder)
            key=(rate,n)
            if key not in grids:
                bx=np.linspace(bounds[0],bounds[1],n+1);by=np.linspace(bounds[2],bounds[3],n+1)
                grids[key]=dict(hist=(1.,bx,by,np.zeros((n,n),np.int64)),path=folder/f'base_{rate:.12g}_n{n}.npz')
            plans.append(dict(base_rate=rate,budget_scope=scope,seed=seed,budget_bytes=int(cap),grid_n=n,
                planned_payload_and_alpha_bytes=planned,next_grid_payload_and_alpha_bytes=next_bytes,
                selection='largest square raw uniform grid fitting actual uint32 NPZ bytes plus 8-byte SA alpha'))
    raw_paths=[]
    for record in data['raw_manifest']:
        name=Path(record['raw_path']).name
        options=[Path(cfg['raw_data_dir'])/name] if cfg['raw_data_dir'] else [Path(record['raw_path']),Path(cfg['repo_root'])/'range_query'/name]
        path=next((p for p in options if p.is_file()),None)
        if path is None:raise FileNotFoundError('Missing matching raw file '+name+'. Set raw_data_dir to its folder; preserve the source15 compressed/uncompressed file identity.')
        raw_paths.append(verified_path(path,record['raw_sha256'],'raw construction input'))
    rows=[];tick=time.perf_counter()
    for i,(path,fine_source) in enumerate(zip(raw_paths,data['raw_hists'])):
        nx,ny=fine_source[3].shape
        parent=dict(bx=fine_source[1][::2],by=fine_source[2][::2],C=fine_source[3].reshape(nx//2,2,ny//2,2).sum(axis=(1,3)))
        check=np.zeros_like(fine_source[3],dtype=np.int64);count=0
        for points in helpers['iter_raw_points'](path):
            ix,iy=helpers['coarse_indices'](points,parent)
            hx=points[:,0]>=(parent['bx'][ix]+(parent['bx'][ix+1]-parent['bx'][ix])/2)
            hy=points[:,1]>=(parent['by'][iy]+(parent['by'][iy+1]-parent['by'][iy])/2)
            check+=np.bincount((2*ix+hx)*ny+(2*iy+hy),minlength=nx*ny).reshape(nx,ny)
            for grid in grids.values():
                _,bx,by,c=grid['hist'];n=c.shape[0]
                if ((points[:,0]<bx[0])|(points[:,0]>=bx[-1])|(points[:,1]<by[0])|(points[:,1]>=by[-1])).any():
                    raise ValueError('Raw point outside fixed base extent; refusing to drop points')
                jx=np.searchsorted(bx,points[:,0],side='right')-1
                jy=np.searchsorted(by,points[:,1],side='right')-1
                c+=np.bincount(jx*n+jy,minlength=n*n).reshape(n,n)
            count+=len(points)
        gap=float(np.max(np.abs(check-fine_source[3])))
        rows.append(dict(source_id=data['raw_manifest'][i]['source_id'],raw_rows=count,raw2_subcell_max_gap=gap,
                         raw_identity_matches=sha(path)==data['raw_manifest'][i]['raw_sha256']))
        pd.DataFrame(rows).to_csv(destination/'finer_raw_parity.csv',index=False)
        if gap!=0 or not rows[-1]['raw_identity_matches']:raise ValueError('Finer construction raw identity/subcell parity failed')
    state['timings']['finer_raw_scan_and_subcell_audit_seconds']=time.perf_counter()-tick
    if sum(r['raw_rows'] for r in rows)!=expected_total:raise ValueError('Finer construction raw total mismatch')
    for key,grid in grids.items():
        if int(grid['hist'][3].sum())!=expected_total:raise ValueError('Finer mass was not conserved')
        tick=time.perf_counter();helpers['save_raw_payload'](grid['path'],grid['hist'])
        grid['hist']=helpers['load_raw_payload'](grid['path'])
        grid['alpha']=sa_alpha(grid['hist']);grid['alpha_path']=grid['path'].with_suffix('.alpha')
        np.asarray([grid['alpha']],dtype='<f8').tofile(grid['alpha_path'])
        grid['bytes']=grid['path'].stat().st_size+grid['alpha_path'].stat().st_size
        grid['setup_seconds']=time.perf_counter()-tick
    for plan in plans:
        grid=grids[(plan['base_rate'],plan['grid_n'])]
        plan.update(actual_payload_and_alpha_bytes=grid['bytes'],unused_budget_bytes=plan['budget_bytes']-grid['bytes'],
            alpha=grid['alpha'],raw_point_count=expected_total,payload_sha256=sha(grid['path']),
            artifact=str(grid['path']),representation='raw-built uniform square histogram; not an ARM export')
        if grid['bytes']!=plan['planned_payload_and_alpha_bytes'] or grid['bytes']>plan['budget_bytes'] or plan['next_grid_payload_and_alpha_bytes']<=plan['budget_bytes']:
            raise ValueError('Actual finer payload violates largest-fitting byte protocol')
    budgets=pd.DataFrame(plans);budgets.to_csv(destination/'budgets.csv',index=False)
    state['finer_grid_builds']=len(grids)
    return grids,budgets

## Evaluate every frozen test query

`fixed_answer_prediction.csv` compares numerical error predictions only for the same original H. `system_quality.csv`
reports actual answer error and the fraction within 25% tolerance for every representation, including changed answers.
`fixed_coverage.csv` uses predeclared 10%/25%/50%/100% diagnostics and retains complete score ties, reporting actual coverage
and selected counts. No test truth chooses a new gate. The full risk-coverage endpoints are also exported.

`auxiliary_estimates.csv` saves H/Sfine/Scoarse and zero-estimate flags so these baselines can be inspected later without
rebuilding features. Source sums remain approximations. Raw SA numerical MAE is blank, never treated as an error percentage.

In [ ]:
def proxy_error(h,reference):
    return abs(float(h)-float(reference))/max(float(reference),1.)


def fixed_coverage(errors,scores):
    scores=np.asarray(scores,float);errors=np.asarray(errors,float);rows=[]
    for fraction in FRACTIONS:
        cut=np.sort(scores,kind='stable')[max(0,math.ceil(fraction*len(scores))-1)]
        mask=scores<=cut;n=int(mask.sum());bad=int((mask&(errors>EPSILON)).sum())
        rows.append(dict(nominal_coverage=fraction,actual_coverage=n/len(scores),accepted_count=n,violation_count=bad,
                         violation=bad/n if n else np.nan,selected_answer_mean_error_percent=float(100*errors[mask].mean()) if n else np.nan,
                         threshold=float(cut),extra_tied_cases=n-math.ceil(fraction*len(scores))))
    return rows


def select_grid(grids,budgets,rate,seed,scope):
    part=budgets[(budgets.base_rate==rate)&(budgets.budget_scope==scope)]
    if scope=='total':part=part[part.seed==seed]
    if len(part)!=1:raise ValueError('Missing/duplicate finer budget route')
    return grids[(rate,int(part.iloc[0].grid_n))]


def evaluate_all(cfg,data,helpers,grids,budgets,destination,state):
    predictions=[];auxiliary=[];audits=[];quality=[];numerical=[];coverage=[];curves=[];payload_rows=[]
    from sklearn.metrics import roc_auc_score
    total_aux=sum(p.stat().st_size for p in data['raw_paths'])
    total_coarse=sum(p.stat().st_size for p in data['coarse_paths'])
    cached={}
    for rate in data['rates']:
        selected=data['saved'][data['saved'].base_rate==rate]
        qids=sorted(set(selected.qId.astype(int)))
        hist=data['base_hists'][rate];alpha=data['alphas'][rate]
        base_answer=[];base_sa=[];s_fine=[];s_coarse=[]
        for qid in qids:
            q=data['queries'][qid,:4]
            h,score=query_answer_sa(q,hist,alpha)
            base_answer.append(h);base_sa.append(score)
            s_fine.append(sum(histogram_count(q,source) for source in data['raw_hists']))
            s_coarse.append(sum(histogram_count(q,source) for source in data['coarse_hists']))
        table=pd.DataFrame(dict(qId=qids,base_answer=base_answer,base_sa=base_sa,s_fine=s_fine,s_coarse=s_coarse))
        table['truth']=data['queries'][qids,4];table['base_rate']=rate
        table['raw2_proxy']=[proxy_error(h,s) for h,s in zip(table.base_answer,table.s_fine)]
        table['coarse_proxy']=[proxy_error(h,s) for h,s in zip(table.base_answer,table.s_coarse)]
        table['s_fine_zero']=table.s_fine==0;table['s_coarse_zero']=table.s_coarse==0
        auxiliary.append(table);cached[rate]=table.set_index('qId')
        # Fine histograms are query-independent and shared across repeated seeds whenever their byte plan matches.
        for key,grid in grids.items():
            if key[0]!=rate:continue
            values=[query_answer_sa(data['queries'][qid,:4],grid['hist'],grid['alpha']) for qid in qids]
            grid['evaluated']=pd.DataFrame(values,index=qids,columns=['answer','score'])
        for seed in cfg['seeds']:
            test_ids=set(data['outer'].loc[(data['outer'].seed==seed)&(data['outer'].split=='test'),'qId'])
            reference=data['saved'][(data['saved'].seed==seed)&(data['saved'].variant=='raw2_source')&(data['saved'].base_rate==rate)].sort_values('qId')
            raw_sa=data['saved'][(data['saved'].seed==seed)&(data['saved'].variant=='sa_raw')&(data['saved'].base_rate==rate)].sort_values('qId')
            if set(reference.qId)!=test_ids or set(raw_sa.qId)!=test_ids:raise ValueError('Source15 predictions do not cover frozen test')
            for record in (reference,raw_sa):
                if not np.allclose(record.relErr,abs(record.estimate-record.truth)/record.truth,rtol=1e-12,atol=1e-12):
                    raise ValueError('Saved source errors disagree with saved answers/truth')
            if not np.allclose(reference.predicted_error,reference.score,rtol=0,atol=1e-12) or (reference.score<0).any():
                raise ValueError('Saved ML score is not the zero-floored relative-error prediction')
            part=cached[rate].loc[reference.qId]
            estimate_gap=float(np.max(np.abs(reference.estimate.to_numpy()-part.base_answer.to_numpy())))
            sa_gap=float(np.max(np.abs(raw_sa.score.to_numpy()-part.base_sa.to_numpy())))
            matches=(estimate_gap<1e-8 and sa_gap<1e-8 and np.array_equal(reference.truth.to_numpy(),part.truth.to_numpy()))
            audits.append(dict(seed=seed,base_rate=rate,max_estimate_gap=estimate_gap,max_sa_gap=sa_gap,matches=bool(matches)))
            pd.DataFrame(audits).to_csv(destination/'base_sa_parity.csv',index=False)
            if not matches:raise ValueError('Live SA/estimate does not reproduce saved15 baseline')
            n=len(part);h=part.base_answer.to_numpy();truth=part.truth.to_numpy()
            if (truth<10).any():raise ValueError('Cohort differs from inherited true count >=10')
            fine_syn=select_grid(grids,budgets,rate,seed,'synopsis')
            fine_total=select_grid(grids,budgets,rate,seed,'total')
            methods={
                'base_sa':(h,part.base_sa.to_numpy(),None,'fixed_H'),
                'raw2_ml':(h,reference.score.to_numpy(),reference.predicted_error.to_numpy(),'fixed_H'),
                'raw2_proxy':(h,part.raw2_proxy.to_numpy(),part.raw2_proxy.to_numpy(),'fixed_H'),
                'coarse_proxy':(h,part.coarse_proxy.to_numpy(),part.coarse_proxy.to_numpy(),'fixed_H'),
                'fine_synopsis_sa':(fine_syn['evaluated'].loc[reference.qId,'answer'].to_numpy(),fine_syn['evaluated'].loc[reference.qId,'score'].to_numpy(),None,'changed_answer'),
                'fine_total_sa':(fine_total['evaluated'].loc[reference.qId,'answer'].to_numpy(),fine_total['evaluated'].loc[reference.qId,'score'].to_numpy(),None,'changed_answer')}
            base_bytes=data['base_paths'][rate].stat().st_size
            for method,(answer,score,pred,view) in methods.items():
                err=np.abs(answer-truth)/truth
                if not np.isfinite(err).all() or not np.isfinite(score).all():raise ValueError('Nonfinite answer/score')
                prefix=dict(seed=seed,method=method,base_rate=rate,comparison_view=view)
                auc=float(roc_auc_score(err>EPSILON,score)) if len(np.unique(err>EPSILON))==2 else np.nan
                quality.append(dict(**prefix,test_count=n,unique_scores=len(np.unique(score)),good_count=int((err<=EPSILON).sum()),
                    good_fraction=float(np.mean(err<=EPSILON)),answer_mean_relative_error_percent=float(100*err.mean()),
                    answer_median_relative_error_percent=float(100*np.median(err)),answer_p95_relative_error_percent=float(100*np.quantile(err,.95)),auc_bad=auc))
                if view=='fixed_H':
                    numerical.append(dict(**prefix,**helpers['prediction_metrics'](err,pred,score)))
                predictions.append(pd.DataFrame(dict(qId=reference.qId.to_numpy(),truth=truth,answer=answer,actual_rel_error=err,
                    score=score,predicted_error=np.nan if pred is None else pred,**prefix)))
                coverage.extend(dict(**prefix,**row) for row in fixed_coverage(err,score))
                curves.extend(dict(**prefix,**row) for row in helpers['risk_coverage_rows'](err,score))
                if method=='base_sa':syn=base_bytes+8;model=0
                elif method=='raw2_ml':syn=base_bytes+total_aux;model=data['model_paths'][seed].stat().st_size
                elif method=='raw2_proxy':syn=base_bytes+total_aux;model=0
                elif method=='coarse_proxy':syn=base_bytes+total_coarse;model=0
                else:syn=fine_syn['bytes'] if method=='fine_synopsis_sa' else fine_total['bytes'];model=0
                payload_rows.append(dict(**prefix,synopsis_bytes=syn,model_bytes=model,total_serialized_bytes=syn+model,
                    process_memory_included=False,sa_alpha_bytes=8 if method.endswith('_sa') else 0))
            # Direct auxiliary answers are quality-only alternatives. No invented SA for a sum of nonaligned source grids.
            for method,answer,syn in [('raw2_direct_answer',part.s_fine.to_numpy(),total_aux),('coarse_direct_answer',part.s_coarse.to_numpy(),total_coarse)]:
                err=np.abs(answer-truth)/truth
                quality.append(dict(seed=seed,method=method,base_rate=rate,comparison_view='direct_auxiliary_answer_no_evaluator',
                    test_count=n,unique_scores=np.nan,good_count=int((err<=EPSILON).sum()),good_fraction=float((err<=EPSILON).mean()),
                    answer_mean_relative_error_percent=float(100*err.mean()),answer_median_relative_error_percent=float(100*np.median(err)),
                    answer_p95_relative_error_percent=float(100*np.quantile(err,.95)),auc_bad=np.nan))
                payload_rows.append(dict(seed=seed,method=method,base_rate=rate,comparison_view='direct_auxiliary_answer_no_evaluator',
                    synopsis_bytes=syn,model_bytes=0,total_serialized_bytes=syn,process_memory_included=False,sa_alpha_bytes=0))
    predictions=pd.concat(predictions,ignore_index=True)
    predictions.to_csv(destination/'per_query.csv',index=False)
    pd.concat(auxiliary,ignore_index=True).to_csv(destination/'auxiliary_estimates.csv',index=False)
    pd.DataFrame(quality).to_csv(destination/'system_quality.csv',index=False)
    pd.DataFrame(numerical).to_csv(destination/'fixed_answer_prediction.csv',index=False)
    pd.DataFrame(coverage).to_csv(destination/'fixed_coverage.csv',index=False)
    pd.DataFrame(curves).to_csv(destination/'risk_coverage.csv',index=False)
    pd.DataFrame(payload_rows).to_csv(destination/'payload_costs.csv',index=False)
    quality=pd.DataFrame(quality)
    quality.groupby(['method','base_rate','comparison_view'],as_index=False).agg(
        seeds=('seed','nunique'),mean_answer_error_percent=('answer_mean_relative_error_percent','mean'),
        mean_good_fraction=('good_fraction','mean'),mean_auc_bad=('auc_bad','mean')).to_csv(destination/'summary.csv',index=False)
    return predictions,cached

## Pair comparisons and measure live request costs

Numeric ML-versus-proxy differences use identical H/true errors. Repeated seed appearances are averaged per query before
resampling inferred source-cell clusters; uncertainty is conditional and exploratory, not independent seed replication.

All arms receive common hash-selected timing queries. Arm order is randomized within paired query/repeat blocks, with
one fixed warm-up per query/arm. Timers call actual estimation/SA/proxy/inference, not a prediction lookup. Scores and
answers are checked AFTER timing. Models and histograms are resident, with no raw reads or fallback. Local payload/context
initialization and model loading are reported separately; filesystem caches are uncontrolled, not cold remote I/O.

In [ ]:
def paired_numeric_effects(predictions,data,destination):
    frames={m:p for m,p in predictions.groupby('method')}
    rows=[]
    for reference in ('raw2_proxy','coarse_proxy'):
        joined=frames['raw2_ml'].merge(frames[reference],on=['seed','base_rate','qId'],suffixes=('_ml','_ref'),validate='one_to_one')
        if not np.allclose(joined.actual_rel_error_ml,joined.actual_rel_error_ref,rtol=0,atol=1e-12):raise ValueError('Different fixed-answer targets')
        joined['delta']=100*(abs(joined.predicted_error_ml-joined.actual_rel_error_ml)-abs(joined.predicted_error_ref-joined.actual_rel_error_ref))
        cells=data['outer'][['qId','cell']].drop_duplicates().set_index('qId').cell
        if cells.index.duplicated().any():raise ValueError('A query has inconsistent cell identities')
        for rate,part in joined.groupby('base_rate'):
            perq=part.groupby('qId').delta.mean().to_frame();perq['cell']=cells.loc[perq.index].astype(str).to_numpy()
            group=perq.groupby('cell').delta.agg(['sum','count']);rng=np.random.default_rng(16001);draws=[]
            if len(group)>1:
                for _ in range(2000):
                    ix=rng.integers(0,len(group),len(group));draws.append(float(group['sum'].to_numpy()[ix].sum()/group['count'].to_numpy()[ix].sum()))
            rows.append(dict(reference=reference,base_rate=rate,unique_queries=len(perq),cell_groups=len(group),
                mean_ml_minus_proxy_mae_pp=float(perq.delta.mean()),ci95_low_pp=float(np.quantile(draws,.025)) if draws else np.nan,
                ci95_high_pp=float(np.quantile(draws,.975)) if draws else np.nan,bootstrap_repeats=len(draws),
                uncertainty_scope='exploratory; repeated seed appearances averaged by query; cell clusters; conditional on fitted models'))
    pd.DataFrame(rows).to_csv(destination/'paired_ml_vs_proxy.csv',index=False)


def runtime_call(q,method,rate,seed,data,namespace,ml_contexts,grids,budgets):
    # This provider receives no truth/labels, saved scores, raw file paths or exact-count lookup.
    if method=='base_sa':return query_answer_sa(q,data['base_hists'][rate],data['alphas'][rate])
    if method in ('raw2_proxy','coarse_proxy'):
        h=histogram_count(q,data['base_hists'][rate])
        sources=data['raw_hists'] if method=='raw2_proxy' else data['coarse_hists']
        auxiliary=sum(histogram_count(q,s) for s in sources)
        return h,proxy_error(h,auxiliary)
    if method.startswith('fine_'):
        grid=grids[(rate,seed,method)]  # route chosen once before any request timer
        return query_answer_sa(q,grid['hist'],grid['alpha'])
    if method=='raw2_ml':
        f=namespace['extract_query'](q,ml_contexts['base'][rate],ml_contexts['merged_helpers'],sources=ml_contexts['sources'])
        f.update({'raw2_'+k:v for k,v in list(f.items()) if k.startswith('src_')})
        array=np.array([[f[n] for n in data['names']]],float)
        return float(f['est_py']),max(float(data['models'][seed].predict(array,num_threads=data['num_threads'])[0]),0.)
    raise ValueError('Unknown runtime method')


def record_runtime(cfg,data,namespace,grids,budgets,predictions,destination,state):
    import sys
    sys.path.insert(0,str(Path(cfg['repo_root'])/'ml'))
    import features
    if Path(features.__file__).resolve().parent!=Path(cfg['repo_root']).resolve()/'ml':raise ValueError('Another checkout is imported; restart Python')
    contexts=dict(base={r:namespace['histogram_context'](h) for r,h in data['base_hists'].items()},
                  sources=[namespace['histogram_context'](h) for h in data['raw_hists']],merged_helpers=features)
    runtime={k:data[k] for k in ('base_hists','alphas','raw_hists','coarse_hists','models','names')}
    runtime['num_threads']=int(state['source15_config_num_threads'])
    runtime_grids={}
    for rate in data['rates']:
        for seed in cfg['seeds']:
            for method,scope in [('fine_synopsis_sa','synopsis'),('fine_total_sa','total')]:
                grid=select_grid(grids,budgets,rate,seed,scope)
                runtime_grids[(rate,seed,method)]=dict(hist=grid['hist'],alpha=grid['alpha'])
    # Measure local reads/context setup separately. OS caches are uncontrolled; these are not cold/remote reads.
    for seed in cfg['seeds']:
        for rate in data['rates']:
            for method in METHODS:
                tick=time.perf_counter()
                if method.startswith('fine_'):
                    grid=select_grid(grids,budgets,rate,seed,'synopsis' if method=='fine_synopsis_sa' else 'total')
                    h=load_runtime_raw(grid['path']);float(np.fromfile(grid['alpha_path'],dtype='<f8')[0])
                else:
                    h=namespace['load_hist_payload'](data['base_paths'][rate])
                    if method=='base_sa':float(np.fromfile(data['alpha_paths'][rate],dtype='<f8')[0])
                    elif method=='coarse_proxy':
                        for p in data['coarse_paths']:namespace['load_hist_payload'](p)
                    else:
                        source_hists=[load_runtime_raw(p) for p in data['raw_paths']]
                        if method=='raw2_ml':
                            namespace['histogram_context'](h)
                            for s in source_hists:namespace['histogram_context'](s)
                state['initialization'].append(dict(method=method,seed=seed,base_rate=rate,kind='local_payload_and_context_setup',seconds=time.perf_counter()-tick))
    expected=predictions.set_index(['seed','base_rate','method','qId'])[['answer','score']]
    records=[];checks=[]
    for seed in cfg['seeds']:
        selected=choose_ids(data['outer'].loc[(data['outer'].seed==seed)&(data['outer'].split=='test'),'qId'],cfg['timing_queries'],f'16-timing:{seed}')
        for rate in data['rates']:
            for qid in selected:
                q=data['queries'][qid,:4]
                for method in METHODS:runtime_call(q,method,rate,seed,runtime,namespace,contexts,runtime_grids,budgets)
            rng=np.random.default_rng(160000+seed)
            # Random paired arm order for every query/repeat, unlike the fixed method order in 15.
            for repeat in range(cfg['timing_repeats']):
                for qid in rng.permutation(selected):
                    for order,method in enumerate(rng.permutation(METHODS)):
                        start=time.perf_counter_ns()
                        answer,score=runtime_call(data['queries'][qid,:4],method,rate,seed,runtime,namespace,contexts,runtime_grids,budgets)
                        stop=time.perf_counter_ns()
                        old=expected.loc[(seed,rate,method,int(qid))]
                        matches=np.isclose(answer,old.answer,rtol=1e-10,atol=1e-8) and np.isclose(score,old.score,rtol=1e-9,atol=1e-8)
                        if not matches:raise ValueError('Live runtime answer/score differs from full-query result: '+str((seed,method,qid)))
                        records.append(dict(seed=seed,base_rate=rate,qId=int(qid),repeat=repeat,method=method,arm_order=order,
                            total_ms=(stop-start)/1e6,cache='resident histograms/contexts/models; one warm-up per query/arm; randomized arm order; no raw reads'))
            print('Timing complete: seed',seed,'base rate',rate,flush=True)
    pd.DataFrame(state['initialization']).to_csv(destination/'initialization.csv',index=False)
    timing=pd.DataFrame(records);timing.to_csv(destination/'resident_timings.csv',index=False)
    timing.groupby(['seed','method','base_rate'],as_index=False).agg(
        measured_requests=('total_ms','size'),unique_queries=('qId','nunique'),mean_ms=('total_ms','mean'),
        p95_ms=('total_ms',lambda s:s.quantile(.95))).to_csv(destination/'resident_timing_summary.csv',index=False)
    checks=timing[['seed','method','base_rate']].drop_duplicates().assign(live_answer_and_score_match=True)
    checks.to_csv(destination/'live_parity.csv',index=False)


def load_runtime_raw(path):
    # Replaced with the verified notebook 15 loader before execution; never a training or query-time raw-point loader.
    raise RuntimeError('Verified payload loader not initialized')

## Outputs and interpretation

The ZIP includes compact CSVs, English notes, provenance and three figures. It excludes raw points, models and payloads.
If same-information proxies match ML, compare their lower cost. If a finer answer is better at the relevant budget,
report that rather than packaging raw2 as superior. Useful candidates still need fresh held-out queries and a new E2E run.

In [ ]:
def write_report(destination):
    import matplotlib.pyplot as plt
    numeric=pd.read_csv(destination/'fixed_answer_prediction.csv')
    quality=pd.read_csv(destination/'system_quality.csv')
    cov=pd.read_csv(destination/'fixed_coverage.csv')
    costs=pd.read_csv(destination/'payload_costs.csv')
    timing=pd.read_csv(destination/'resident_timing_summary.csv')
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    a=numeric.groupby('method').mae_pp.mean().reindex(['raw2_ml','raw2_proxy','coarse_proxy'])
    axes[0].bar(a.index,a);axes[0].set(ylabel='Error-prediction MAE (percentage points)',title='Same original histogram answer')
    b=quality.groupby('method').good_fraction.mean()
    axes[1].bar(b.index,100*b);axes[1].set(ylabel='Answers within 25% error (%)',title='Actual answer quality; answers may differ')
    for ax in axes:ax.tick_params(axis='x',rotation=30)
    fig.tight_layout();fig.savefig(destination/'quality_views.png',dpi=150);plt.close(fig)
    fig,ax=plt.subplots(figsize=(8,5))
    for method,part in cov.groupby('method'):
        avg=part.groupby('nominal_coverage')[['actual_coverage','violation']].mean()
        ax.plot(avg.actual_coverage,avg.violation,marker='o',label=method)
    ax.set(xlabel='Actual selected fraction (complete ties)',ylabel='Violation fraction among selected answers',
           title='Same queries; fine methods return different answers')
    ax.legend();fig.tight_layout();fig.savefig(destination/'risk_coverage.png',dpi=150);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for method in METHODS:
        rows=costs[costs.method==method];time_rows=timing[timing.method==method]
        meanq=quality[quality.method==method].good_fraction.mean()
        axes[0].scatter(rows.total_serialized_bytes.mean()/1024,100*meanq,label=method)
        axes[1].scatter(time_rows.mean_ms.mean(),100*meanq,label=method)
    axes[0].set(xlabel='Synopsis + model bytes (KiB)',ylabel='All-query good-answer fraction (%)')
    axes[1].set(xlabel='Mean resident request time (ms)',ylabel='All-query good-answer fraction (%)')
    axes[1].legend(fontsize=8);fig.suptitle('System tradeoffs; no fallback or E2E speedup measured')
    fig.tight_layout();fig.savefig(destination/'cost_quality.png',dpi=150);plt.close(fig)
    text='''# Information versus model versus a finer answer

This is a no-training, no-gate-retuning exploratory comparison using the exact saved notebook 15 outer tests. Predictions of the fixed original answer are separated from systems that return a different finer answer.

## Fixed-answer numerical prediction

'''+numeric.groupby('method')[['mae_pp','p90_ae_pp','bias_pp','auc_bad']].mean().round(5).to_csv()+'''
Raw SA is a ranking score, so its numerical error MAE remains undefined. Proxy formulas are fixed: abs(H-S)/max(S,1), using either raw2 or coarse source estimates. Zero auxiliary answers are exported explicitly; this proxy is not a certified bound. No test outcome chooses a formula, resolution or operating cutoff.

## Actual answer quality

'''+quality.groupby('method')[['answer_mean_relative_error_percent','good_fraction','auc_bad']].mean().round(5).to_csv()+'''
The first four methods return the SAME original H. Their actual answer quality must be identical. Fine methods return different answers and have different true-error labels; do not rank their numerical prediction MAE against fixed-H models. The direct auxiliary-answer alternatives have no invented SA score or ranking curve.

## Budget rules

The synopsis cap is the actual original base plus all raw2 payload bytes. The total cap additionally includes the matching saved model file bytes for each seed. Each finer baseline is a NEW uniform square histogram counted from the SAME raw points, not a finer ARM export. It uses the same uncompressed uint32-count payload format as raw2. The largest fitting square resolution is selected by actual file bytes plus an 8-byte persisted SA alpha, without queries or truth. budgets.csv reports slack and the next infeasible grid. Model weights, derived arrays and process memory are separate; total serialized bytes are not process-memory measurements.

SA follows the repository beta/MAD formula and area-fraction-weighted query score, including empty cells. The original base alpha comes from the verified original logger and must reproduce saved15 scores. New square grids avoid the legacy rectangular flattening issue; all-zero median/MAD and resulting tied SA scores are retained.

## Scope and next decision

If the fine proxy matches ML, its lower model/feature cost may be preferable; this is evidence about whether ML adds value beyond the new information. If an equal-budget finer answer dominates the relevant quality/cost tradeoff, do not present raw2 as a better system solely because its fixed-H prediction MAE improved. A quality advantage worth preserving would motivate smaller representations/models and fresh-query validation.

All test queries, including answers not selected by ranking diagnostics, are evaluated. Complete score ties stay together; actual coverage can exceed nominal. These test-score cutoffs are descriptions, not deployment gates. The historical 5% accepted-risk target is not imposed. Zero acceptance has undefined risk. The inherited cohort excludes exact counts below 10, including zero.

Repeated seed/rate appearances are not independent queries. paired_ml_vs_proxy.csv averages repeated seed appearances per query before cell-cluster resampling, conditional on fitted models; it does not establish strict geographic or cross-dataset generalization. Existing tests were already inspected, so all conclusions remain exploratory.

## Cost scope

Query timings call live estimate/score/prediction providers and verify saved parity AFTER each timer. They do not look up a saved answer or read raw points inside the request timer. The simple proxy uses count-only overlap operations, not all seven ML source features. Arms share selected queries and use randomized arm order with fixed warm-up. Contexts/models are resident; local payload/context initialization, model loading, finer raw construction and hashing are recorded separately. Filesystem caches are uncontrolled; no cold remote I/O, exact fallback or E2E speedup is claimed. Existing notebook 15/09 timings are not substituted for these measurements.
'''
    (destination/'analysis_en.md').write_text(text)


def export_results(destination):
    with ZipFile(destination/'information_budget_results.zip','w',ZIP_DEFLATED) as archive:
        for path in sorted(destination.iterdir()):
            if path.is_file() and path.suffix in {'.csv','.json','.md','.png'}:
                archive.write(path,path.name)


def run_comparison(cfg):
    global lgb,uniform_filter,roc_auc_score,load_runtime_raw
    import lightgbm as lgb
    from scipy.ndimage import uniform_filter
    from sklearn.metrics import roc_auc_score
    destination=Path(cfg['output_root']).expanduser().resolve()/('information-budget-'+uuid4().hex[:12])
    destination.mkdir(parents=True,exist_ok=False)
    state=dict(status='running',study='information-budget-comparison',config=cfg,timings={},initialization=[],
        started_utc=datetime.now(timezone.utc).isoformat(),python=platform.python_version(),epsilon=EPSILON,
        protocol=dict(training=False,gate_retuning=False,acceptance_risk_target=None,fixed_coverages=list(FRACTIONS),
            proxy='abs(H-S)/max(S,1)',finer='largest byte-feasible raw-built uniform square grid; same uint32 payload format',
            budget_scopes=['synopsis','synopsis_plus_saved_model'],exploratory=True))
    save_json(destination/'run.json',state);started=time.perf_counter()
    print('Writing new comparison:',destination,flush=True)
    try:
        source,prior=resolve_source(cfg)
        helpers,namespace,provenance=load_verified_helpers(cfg['repo_root']);state['reuse']=provenance
        load_runtime_raw=helpers['load_raw_payload']
        data=load_inputs(cfg,source,prior,destination,helpers,namespace,state)
        state['source15_config_num_threads']=int(prior['inherited_training_config']['num_threads'])
        save_json(destination/'run.json',state)
        grids,budgets=build_finer_baselines(cfg,data,destination,helpers,namespace,state)
        save_json(destination/'run.json',state)
        tick=time.perf_counter()
        predictions,_=evaluate_all(cfg,data,helpers,grids,budgets,destination,state)
        state['timings']['full_test_evaluation_seconds']=time.perf_counter()-tick
        paired_numeric_effects(predictions,data,destination)
        record_runtime(cfg,data,namespace,grids,budgets,predictions,destination,state)
        write_report(destination)
        state.update(status='complete',completed_utc=datetime.now(timezone.utc).isoformat(),
            total_seconds=time.perf_counter()-started,per_query_rows=len(predictions))
        save_json(destination/'run.json',state);export_results(destination)
        print('Completed:',destination,flush=True)
        print('Share:',destination/'information_budget_results.zip',flush=True)
        return destination
    except Exception as error:
        state.update(status='failed',error_type=type(error).__name__,error=str(error),total_seconds=time.perf_counter()-started)
        save_json(destination/'run.json',state);export_results(destination)
        print('Failed run diagnostics:',destination/'information_budget_results.zip',flush=True)
        raise

## Execute the comparison

This cell runs comparisons only; there is no call to `fit`. It creates a new result directory. Share
**`information_budget_results.zip`**, including a failed-run ZIP if a parity/input check stops execution.

In [ ]:
# RUN_BUDGET_COMPARISON
config = load_settings()
result_dir = run_comparison(config)
print(pd.read_csv(result_dir / 'summary.csv').to_string(index=False))